In [ ]:
# Step 1: Importing Necessary Libraries & CSV File

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set up visualization style
plt.style.use('seaborn-v0_8')
%matplotlib inline

# Load your data
df = pd.read_csv('RentingOutofFlats2025.csv')  # Replace with your actual file path

# Initial exploration
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
df.head()


In [ ]:
# Step 2: Checking Initial Strucutre of the Data

# Check data types for each column
print("DATA TYPES FOR EACH COLUMN:")
print("=" * 40)
print(df.dtypes)
print("\n" + "=" * 40)

# More detailed information about each column
print("\nDETAILED COLUMN INFORMATION:")
print("=" * 40)
for column in df.columns:
    print(f"\nColumn: {column}")
    print(f"Data type: {df[column].dtype}")
    print(f"Number of unique values: {df[column].nunique()}")
    print(f"Sample values: {df[column].head(3).tolist()}")


In [ ]:
# Step 3: Data Cleaning & Pre-processing

# Make a copy of the original dataframe to preserve raw data
df_clean = df.copy()

print("STARTING DATA TYPE CONVERSIONS...")
print("=" * 50)

# Convert rent_approval_date to datetime
print("1. Converting rent_approval_date to datetime...")
df_clean['rent_approval_date'] = pd.to_datetime(df_clean['rent_approval_date'])
print(f"   New dtype: {df_clean['rent_approval_date'].dtype}")
print(f"   Date range: {df_clean['rent_approval_date'].min()} to {df_clean['rent_approval_date'].max()}")

# Convert town to category (since it has limited unique values)
print("\n2. Converting town to category...")
df_clean['town'] = df_clean['town'].astype('category')
print(f"   New dtype: {df_clean['town'].dtype}")
print(f"   Number of unique towns: {df_clean['town'].nunique()}")

# Convert flat_type to category
print("\n3. Converting flat_type to category...")
df_clean['flat_type'] = df_clean['flat_type'].astype('category')
print(f"   New dtype: {df_clean['flat_type'].dtype}")
print(f"   Unique flat types: {df_clean['flat_type'].cat.categories.tolist()}")

# Check block column for any issues
print("\n4. Checking block column...")
# Remove any extra whitespace
df_clean['block'] = df_clean['block'].str.strip()
print(f"   Block dtype remains: {df_clean['block'].dtype}")
print(f"   Sample blocks: {df_clean['block'].head(5).tolist()}")

# Check street_name column
print("\n5. Checking street_name column...")
# Remove any extra whitespace
df_clean['street_name'] = df_clean['street_name'].str.strip()
print(f"   Street name dtype remains: {df_clean['street_name'].dtype}")
print(f"   Sample streets: {df_clean['street_name'].head(3).tolist()}")

# monthly_rent is already int64 - just verify it looks reasonable
print("\n6. Verifying monthly_rent column...")
print(f"   monthly_rent dtype: {df_clean['monthly_rent'].dtype}")
print(f"   Rent range: ${df_clean['monthly_rent'].min()} to ${df_clean['monthly_rent'].max()}")


In [ ]:
# Step 4: Checking Pre-processing

print("\n" + "=" * 50)
print("FINAL DATA TYPES AFTER CONVERSION:")
print("=" * 50)
print(df_clean.dtypes)

print("\nMEMORY USAGE COMPARISON:")
print(f"Original: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Cleaned:  {df_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nFirst few rows of cleaned data:")
df_clean.head()


In [ ]:
# Step 5: Statistical Analysis

print("COMPREHENSIVE STATISTICAL ANALYSIS")
print("=" * 60)

# Basic statistics for numerical columns
print("1. BASIC STATISTICS FOR MONTHLY RENT:")
print("-" * 40)
rent_stats = df_clean['monthly_rent'].describe()
print(rent_stats)

print("\n2. RENT STATISTICS BY FLAT TYPE:")
print("-" * 40)
rent_by_flat_type = df_clean.groupby('flat_type')['monthly_rent'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
rent_by_flat_type = rent_by_flat_type.round(2)
print(rent_by_flat_type)

print("\n3. TOP 10 TOWNS BY RENTAL VOLUME AND PRICE:")
print("-" * 40)
town_stats = df_clean.groupby('town')['monthly_rent'].agg(['count', 'mean', 'median']).round(2)
town_stats = town_stats.sort_values('count', ascending=False)
print("Top 10 towns by number of rentals:")
print(town_stats.head(10))

print("\n4. YEARLY RENT TRENDS:")
print("-" * 40)
# Extract year from the datetime
df_clean['year'] = df_clean['rent_approval_date'].dt.year
yearly_stats = df_clean.groupby('year')['monthly_rent'].agg(['count', 'mean', 'median', 'std']).round(2)
print(yearly_stats)

print("\n5. DISTRIBUTION ANALYSIS:")
print("-" * 40)
# Calculate percentiles
percentiles = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
percentile_values = df_clean['monthly_rent'].quantile(percentiles)
print("Rent Percentiles:")
for p, value in zip(percentiles, percentile_values):
    print(f"  {p*100:.0f}th percentile: ${value:.0f}")

# Skewness and Kurtosis
from scipy.stats import skew, kurtosis
print(f"\nSkewness: {skew(df_clean['monthly_rent']):.3f}")
print(f"Kurtosis: {kurtosis(df_clean['monthly_rent']):.3f}")

print("\n6. MONTHLY TRENDS (if multi-year data):")
print("-" * 40)
df_clean['month'] = df_clean['rent_approval_date'].dt.month
monthly_trends = df_clean.groupby('month')['monthly_rent'].mean().round(2)
print("Average rent by month:")
print(monthly_trends)

print("\n7. CORRELATION ANALYSIS:")
print("-" * 40)
# Since we have limited numerical variables, let's check correlation with time
df_clean['date_numeric'] = df_clean['rent_approval_date'].astype('int64') // 10**9  # Convert to seconds
correlation = df_clean[['monthly_rent', 'date_numeric']].corr().iloc[0,1]
print(f"Correlation between rent and time: {correlation:.3f}")

print("\n8. OUTLIER ANALYSIS:")
print("-" * 40)
Q1 = df_clean['monthly_rent'].quantile(0.25)
Q3 = df_clean['monthly_rent'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_clean[(df_clean['monthly_rent'] < lower_bound) | (df_clean['monthly_rent'] > upper_bound)]
print(f"Number of potential outliers: {len(outliers)}")
print(f"Outlier range: <${lower_bound:.0f} or >${upper_bound:.0f}")
print(f"Percentage of outliers: {(len(outliers)/len(df_clean)*100):.2f}%")

print("\n9. KEY INSIGHTS SUMMARY:")
print("-" * 40)
print(f"• Total rental records: {len(df_clean):,}")
print(f"• Average monthly rent: ${df_clean['monthly_rent'].mean():.2f}")
print(f"• Median monthly rent: ${df_clean['monthly_rent'].median():.2f}")
print(f"• Most common flat type: {df_clean['flat_type'].value_counts().index[0]}")
print(f"• Town with most rentals: {df_clean['town'].value_counts().index[0]}")
print(f"• Data coverage: {df_clean['rent_approval_date'].min().strftime('%Y-%m')} to {df_clean['rent_approval_date'].max().strftime('%Y-%m')}")
print(f"• Rent standard deviation: ${df_clean['monthly_rent'].std():.2f}")


In [ ]:
# Step 6: Importing Libraries for Data Visualisation

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.offline as pyo
pyo.init_notebook_mode()

print("CREATING INTERACTIVE VISUALIZATIONS WITH PLOTLY...")


In [ ]:
# Step 7: Making Basic Interactive Maps

# 1. Distribution of Monthly Rent
fig1 = px.histogram(df_clean, x='monthly_rent', 
                   title='Distribution of Monthly Rent',
                   labels={'monthly_rent': 'Monthly Rent (S$)'},
                   nbins=50,
                   color_discrete_sequence=['#636EFA'])

fig1.update_layout(
    xaxis_title='Monthly Rent (S$)',
    yaxis_title='Number of Rentals',
    showlegend=False,
    template='plotly_white'
)
fig1.show()

# 2. Average Rent by Flat Type
avg_rent_by_type = df_clean.groupby('flat_type')['monthly_rent'].mean().sort_values(ascending=True)

fig2 = px.bar(x=avg_rent_by_type.values, y=avg_rent_by_type.index,
              title='Average Monthly Rent by Flat Type',
              labels={'x': 'Average Monthly Rent (S$)', 'y': 'Flat Type'},
              color=avg_rent_by_type.values,
              color_continuous_scale='Viridis')

fig2.update_layout(
    template='plotly_white',
    showlegend=False
)
fig2.show()

# 3. Top Towns Analysis
town_summary = df_clean.groupby('town').agg({
    'monthly_rent': ['count', 'mean']
}).round(2)
town_summary.columns = ['rental_count', 'avg_rent']
town_summary = town_summary.sort_values('rental_count', ascending=False).head(15)

fig3 = make_subplots(specs=[[{"secondary_y": True}]])

# Bar chart for rental count
fig3.add_trace(
    go.Bar(x=town_summary.index, y=town_summary['rental_count'], 
           name="Rental Count", marker_color='lightblue'),
    secondary_y=False,
)

# Line chart for average rent
fig3.add_trace(
    go.Scatter(x=town_summary.index, y=town_summary['avg_rent'], 
               name="Avg Rent", line=dict(color='red', width=3)),
    secondary_y=True,
)

fig3.update_layout(
    title_text="Top 15 Towns: Rental Volume vs Average Rent",
    template='plotly_white'
)
fig3.update_xaxes(title_text="Town")
fig3.update_yaxes(title_text="Rental Count", secondary_y=False)
fig3.update_yaxes(title_text="Average Rent (S$)", secondary_y=True)

fig3.show()

# 4. Monthly Rent Trends Over Time
monthly_trends = df_clean.groupby(df_clean['rent_approval_date'].dt.to_period('M'))['monthly_rent'].mean().reset_index()
monthly_trends['rent_approval_date'] = monthly_trends['rent_approval_date'].dt.to_timestamp()

fig4 = px.line(monthly_trends, x='rent_approval_date', y='monthly_rent',
               title='Monthly Rent Trends Over Time',
               labels={'rent_approval_date': 'Date', 'monthly_rent': 'Average Monthly Rent (S$)'})

fig4.update_layout(
    template='plotly_white',
    xaxis=dict(tickformat='%b %Y')
)
fig4.show()

# 5. Rent Distribution by Flat Type (Box Plot)
fig5 = px.box(df_clean, x='flat_type', y='monthly_rent',
              title='Rent Distribution by Flat Type',
              labels={'flat_type': 'Flat Type', 'monthly_rent': 'Monthly Rent (S$)'},
              color='flat_type')

fig5.update_layout(
    template='plotly_white',
    showlegend=False,
    xaxis={'categoryorder':'total descending'}
)
fig5.show()

# 6. Heatmap of Average Rent by Town
town_rent_heatmap = df_clean.groupby('town')['monthly_rent'].mean().sort_values(ascending=False)

fig6 = px.bar(town_rent_heatmap.head(20), 
              x=town_rent_heatmap.head(20).values,
              y=town_rent_heatmap.head(20).index,
              title='Top 20 Towns by Average Rent',
              labels={'x': 'Average Monthly Rent (S$)', 'y': 'Town'},
              color=town_rent_heatmap.head(20).values,
              color_continuous_scale='thermal')

fig6.update_layout(
    template='plotly_white',
    showlegend=False
)
fig6.show()

# 7. Flat Type Distribution
flat_type_dist = df_clean['flat_type'].value_counts()

fig7 = px.pie(values=flat_type_dist.values, 
              names=flat_type_dist.index,
              title='Distribution of Flat Types',
              color_discrete_sequence=px.colors.sequential.RdBu)

fig7.update_traces(textposition='inside', textinfo='percent+label')
fig7.show()

# 8. Year-over-Year Comparison
yearly_comparison = df_clean.groupby('year')['monthly_rent'].agg(['mean', 'count']).reset_index()

fig8 = make_subplots(rows=2, cols=1, 
                    subplot_titles=('Average Rent by Year', 'Number of Rentals by Year'))

fig8.add_trace(
    go.Bar(x=yearly_comparison['year'], y=yearly_comparison['mean'],
           name='Avg Rent', marker_color='coral'),
    row=1, col=1
)

fig8.add_trace(
    go.Bar(x=yearly_comparison['year'], y=yearly_comparison['count'],
           name='Rental Count', marker_color='lightseagreen'),
    row=2, col=1
)

fig8.update_layout(height=600, title_text="Year-over-Year Rental Analysis", template='plotly_white')
fig8.update_yaxes(title_text="Average Rent (S$)", row=1, col=1)
fig8.update_yaxes(title_text="Number of Rentals", row=2, col=1)
fig8.update_xaxes(title_text="Year", row=2, col=1)

fig8.show()


In [ ]:
# Step 8: Saving all Charts

# Save all charts as HTML files for easy inclusion in PowerPoint
fig1.write_html("rent_distribution.html")
fig2.write_html("rent_by_flat_type.html")
fig3.write_html("top_towns_analysis.html")
fig4.write_html("rent_trends_over_time.html")
fig5.write_html("box_plot_by_flat_type.html")
fig6.write_html("top_towns_by_rent.html")
fig7.write_html("flat_type_distribution.html")
fig8.write_html("yearly_comparison.html")

print("All charts have been saved as HTML files! You can open them in a browser and screenshot for PowerPoint.")


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Create pie chart for flat type distribution
flat_type_counts = df_clean['flat_type'].value_counts()

# Calculate percentages
total_transactions = len(df_clean)
percentages = (flat_type_counts / total_transactions * 100).round(1)

# Create the pie chart
fig_pie = go.Figure(data=[go.Pie(
    labels=flat_type_counts.index,
    values=flat_type_counts.values,
    hole=0.5,  # Creates the donut chart effect
    textinfo='label+percent',
    textposition='outside',
    marker=dict(colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']),
    hovertemplate='<b>%{label}</b><br>%{percent} (%{value:,} transactions)<extra></extra>'
)])

# Update layout to match the style
fig_pie.update_layout(
    title={
        'text': f"<b>{total_transactions:,} TRANSACTIONS</b><br>"
                f"<span style='font-size:14px; color:gray'>"
                f"{df_clean['rent_approval_date'].min().strftime('%b %Y') if not df_clean['rent_approval_date'].isna().all() else 'START'} - "
                f"{df_clean['rent_approval_date'].max().strftime('%b %Y') if not df_clean['rent_approval_date'].isna().all() else 'END'}</span>",
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 20}
    },
    showlegend=False,
    annotations=[
        dict(
            text=f"<b>{total_transactions:,}</b><br>TRANSACTIONS",
            x=0.5, y=0.5,
            font_size=16,
            showarrow=False,
            font_color='black'
        )
    ],
    height=600,
    margin=dict(t=100, b=50, l=50, r=50)
)

# Customize the text font and appearance
fig_pie.update_traces(
    textfont=dict(size=14, color='black'),
    texttemplate='%{label}<br>%{percent}%'
)

fig_pie.show()

# Save as PNG
fig_pie.write_image(PLOTS_DIR / "flat_type_pie_chart.png", scale=2)
print("✅ Pie chart downloaded: flat_type_pie_chart.png")


In [ ]:
# Step 9: Town/Avenue/Street EDA - ULTRA FAST VERSION

print("Creating FAST town/street/block visualizations...")

# PRE-COMPUTE all aggregations once (this is the key to speed)
print("Pre-computing aggregations...")

# Town-level stats (fast)
town_stats = df_clean.groupby('town').agg({
    'monthly_rent': ['mean', 'median', 'count'],
    'street_name': 'nunique'
}).round(2)
town_stats.columns = ['avg_rent', 'median_rent', 'rental_count', 'unique_streets']
town_stats = town_stats.reset_index()

# Sample data for complex visualizations (dramatically speeds up plotting)
sample_df = df_clean.sample(n=min(2000, len(df_clean)), random_state=42)

print("Creating visualizations...")

# 1. FAST: Interactive Bar Chart instead of Sunburst
fig1 = px.bar(town_stats.nlargest(20, 'rental_count'),
              x='town', y='rental_count',
              color='avg_rent',
              title='Top 20 Towns by Rental Volume<br><sub>Color shows average rent</sub>',
              color_continuous_scale='Viridis',
              hover_data=['avg_rent', 'median_rent', 'unique_streets'])
fig1.update_layout(template='plotly_white', xaxis_tickangle=45)
fig1.show()

# 2. FAST: Scatter Plot with Town Selection
fig2 = px.scatter(town_stats, 
                  x='rental_count', y='avg_rent',
                  size='unique_streets',
                  color='avg_rent',
                  hover_name='town',
                  size_max=40,
                  title='Town Analysis: Rent vs Rental Volume<br><sub>Bubble size = number of unique streets</sub>',
                  labels={'rental_count': 'Number of Rentals', 'avg_rent': 'Average Rent (S$)'},
                  color_continuous_scale='thermal')
fig2.update_layout(template='plotly_white')
fig2.show()

# 3. FAST: Heatmap - Top Towns vs Flat Types
heatmap_data = df_clean.groupby(['town', 'flat_type'])['monthly_rent'].mean().unstack().round(0)
top_15_towns = town_stats.nlargest(15, 'rental_count')['town']
heatmap_data_top = heatmap_data.loc[heatmap_data.index.intersection(top_15_towns)]

fig3 = px.imshow(heatmap_data_top,
                title='Average Rent Heatmap: Top Towns vs Flat Types',
                color_continuous_scale='Viridis',
                aspect='auto',
                labels={'x': 'Flat Type', 'y': 'Town', 'color': 'Avg Rent (S$)'})
fig3.update_layout(height=500)
fig3.show()

# 4. FAST: Interactive Drill-Down (Corrected Orientation)
print("Creating fast interactive drill-down...")

# Pre-aggregate street data for top towns
top_towns = town_stats.nlargest(8, 'rental_count')['town'].tolist()
town_street_data = {}

for town in top_towns:
    town_data = df_clean[df_clean['town'] == town]
    # Fast aggregation - just take top 10 streets
    street_stats = town_data.groupby('street_name')['monthly_rent'].agg(['mean', 'count']).round(2)
    street_stats = street_stats.nlargest(10, 'count')
    town_street_data[town] = street_stats

# Create interactive figure
fig4 = go.Figure()

for town in top_towns:
    street_stats = town_street_data[town]
    fig4.add_trace(
        go.Bar(y=street_stats['mean'].values,  # CHANGED: y-axis for rent values
               x=street_stats.index,           # CHANGED: x-axis for street names
               name=town,
               orientation='v',                # CHANGED: vertical orientation
               visible=(town == top_towns[0]),
               hovertemplate='<b>%{x}</b><br>Avg Rent: $%{y:.0f}<br>Rentals: %{customdata}',
               customdata=street_stats['count'].values)
    )

# Dropdown menu
dropdown_buttons = []
for i, town in enumerate(top_towns):
    dropdown_buttons.append(
        dict(
            label=town,
            method="update",
            args=[{"visible": [j == i for j in range(len(top_towns))]},
                  {"title": f"Top Streets by Average Rent in {town}"}]
        )
    )

fig4.update_layout(
    updatemenus=[dict(buttons=dropdown_buttons, direction="down", x=0.1, y=1.15)],
    title=f"Top Streets by Average Rent in {top_towns[0]}",
    yaxis_title="Average Monthly Rent (S$)",  # CHANGED: y-axis label
    xaxis_title="Street Name",                # CHANGED: x-axis label
    template='plotly_white',
    height=500,
    xaxis_tickangle=-45  # Rotate street names for better readability
)
fig4.show()

# 5. FAST: Simple Treemap with sampled data
try:
    # Use sampled data for treemap
    treemap_sample = sample_df.groupby(['town', 'street_name']).agg({
        'monthly_rent': 'mean',
        'block': 'count'
    }).reset_index()
    treemap_sample = treemap_sample.rename(columns={'block': 'rental_count'})
    treemap_sample = treemap_sample[treemap_sample['rental_count'] > 0]

    fig5 = px.treemap(treemap_sample,
                     path=['town', 'street_name'],
                     values='rental_count',
                     color='monthly_rent',
                     color_continuous_scale='Viridis',
                     title='Town → Street Hierarchy (Sampled Data)')
    fig5.update_layout(template='plotly_white', height=500)
    fig5.show()
    print("✓ Treemap created successfully!")
except Exception as e:
    print(f"✗ Treemap skipped: {e}")

# 6. FAST: Comparative Analysis - Top 5 Towns
print("Creating comparative analysis...")
top_5_towns = town_stats.nlargest(5, 'rental_count')['town'].tolist()

fig6 = go.Figure()

for town in top_5_towns:
    town_data = df_clean[df_clean['town'] == town]
    rent_hist = town_data['monthly_rent']
    
    fig6.add_trace(
        go.Violin(y=rent_hist, 
                 name=town,
                 box_visible=True,
                 meanline_visible=True)
    )

fig6.update_layout(
    title='Rent Distribution Comparison: Top 5 Towns',
    yaxis_title='Monthly Rent (S$)',
    template='plotly_white',
    height=500
)
fig6.show()

# 7. FAST: Summary Dashboard
print("Creating summary dashboard...")

# Create subplots
from plotly.subplots import make_subplots

fig7 = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Top 10 Towns by Rental Count', 
                   'Average Rent by Town Size',
                   'Town vs Flat Type Distribution',
                   'Rental Volume Trends'),
    specs=[[{"type": "bar"}, {"type": "scatter"}],
          [{"type": "heatmap"}, {"type": "bar"}]]
)

# Plot 1: Top towns bar chart
top_towns_bar = town_stats.nlargest(10, 'rental_count')
fig7.add_trace(
    go.Bar(x=top_towns_bar['town'], y=top_towns_bar['rental_count'],
           name='Rental Count', marker_color='lightblue'),
    row=1, col=1
)

# Plot 2: Rent vs Volume scatter
fig7.add_trace(
    go.Scatter(x=town_stats['rental_count'], y=town_stats['avg_rent'],
               mode='markers', name='Towns',
               marker=dict(size=8, color=town_stats['avg_rent'],
                          colorscale='Viridis', showscale=True)),
    row=1, col=2
)

# Plot 3: Simple town-flat type count heatmap
town_flat_count = pd.crosstab(df_clean['town'], df_clean['flat_type'])
top_10_towns_flat = town_flat_count.loc[town_flat_count.sum(axis=1).nlargest(10).index]
fig7.add_trace(
    go.Heatmap(z=top_10_towns_flat.values,
               x=top_10_towns_flat.columns,
               y=top_10_towns_flat.index,
               colorscale='Blues'),
    row=2, col=1
)

# Plot 4: Monthly trends for top 3 towns
for i, town in enumerate(top_5_towns[:3]):
    town_time = df_clean[df_clean['town'] == town]
    monthly_trend = town_time.groupby(town_time['rent_approval_date'].dt.to_period('M'))['monthly_rent'].count()
    monthly_trend.index = monthly_trend.index.astype(str)
    
    fig7.add_trace(
        go.Scatter(x=monthly_trend.index, y=monthly_trend.values,
                  name=town, mode='lines+markers'),
        row=2, col=2
    )

fig7.update_layout(height=800, title_text="Town Analysis Dashboard", template='plotly_white')
fig7.show()

print("\n" + "="*50)
print("FAST TOWN/STREET ANALYSIS COMPLETED!")
print("✓ 7 interactive charts created in minimal time")
print("="*50)


In [ ]:
spatial_df = df_clean.copy()


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# Prepare the data for year-over-year analysis
print("Preparing year-over-year rental data...")

# Extract year from rent_approval_date
spatial_df['year'] = spatial_df['rent_approval_date'].dt.year

# Filter for complete years (remove partial years if needed)
yearly_data = spatial_df.groupby(['year', 'flat_type'])['monthly_rent'].agg(['mean', 'count']).reset_index()
yearly_data.columns = ['year', 'flat_type', 'avg_rent', 'transaction_count']

# Remove years with insufficient data
yearly_data = yearly_data[yearly_data['transaction_count'] >= 10]  # Minimum transactions per year

# Create a color palette that's distinct and beautiful
color_palette = {
    '1-ROOM': '#E74C3C',      # Vibrant Red
    '2-ROOM': '#3498DB',      # Bright Blue
    '3-ROOM': '#2ECC71',      # Green
    '4-ROOM': '#F39C12',      # Orange
    '5-ROOM': '#9B59B6',      # Purple
    'EXECUTIVE': '#1ABC9C'    # Teal
}

# Ensure all flat types are represented (fill missing years with NaN)
all_years = sorted(yearly_data['year'].unique())
all_flat_types = yearly_data['flat_type'].unique()

# Create complete grid
complete_data = []
for year in all_years:
    for flat_type in all_flat_types:
        complete_data.append({'year': year, 'flat_type': flat_type})

complete_df = pd.DataFrame(complete_data)
yearly_complete = pd.merge(complete_df, yearly_data, on=['year', 'flat_type'], how='left')

# Create the line chart
fig = go.Figure()

# Add lines for each flat type
for flat_type in sorted(yearly_complete['flat_type'].unique()):
    flat_data = yearly_complete[yearly_complete['flat_type'] == flat_type].sort_values('year')
    
    # Only plot if we have at least 2 data points
    if len(flat_data) >= 2:
        fig.add_trace(go.Scatter(
            x=flat_data['year'],
            y=flat_data['avg_rent'],
            mode='lines+markers',
            name=flat_type,
            line=dict(
                width=3,
                color=color_palette.get(flat_type, '#000000')
            ),
            marker=dict(
                size=8,
                color=color_palette.get(flat_type, '#000000'),
                line=dict(width=2, color='white')
            ),
            hovertemplate=(
                f"<b>{flat_type}</b><br>"
                "Year: %{x}<br>"
                "Avg Rent: $%{y:,.0f}<br>"
                "Transactions: %{customdata:,}<extra></extra>"
            ),
            customdata=flat_data['transaction_count']
        ))

# Update layout for beautiful appearance
fig.update_layout(
    title=dict(
        text="<b>Year-over-Year Average Rent by Flat Type</b>",
        x=0.5,
        xanchor='center',
        font=dict(size=24, color='#2C3E50')
    ),
    xaxis=dict(
        title="Year",
        tickmode='array',
        tickvals=all_years,
        gridcolor='lightgray',
        gridwidth=1,
        title_font=dict(size=14, color='#2C3E50')
    ),
    yaxis=dict(
        title="Average Monthly Rent (S$)",
        tickformat="$,.0f",
        gridcolor='lightgray',
        gridwidth=1,
        title_font=dict(size=14, color='#2C3E50')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=700,
    margin=dict(l=80, r=80, t=100, b=80),
    hovermode='closest',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='lightgray',
        borderwidth=1
    )
)

# Add some styling enhancements
fig.update_xaxes(showline=True, linewidth=2, linecolor='gray')
fig.update_yaxes(showline=True, linewidth=2, linecolor='gray')

# Add annotation for data source
fig.add_annotation(
    x=1, y=-0.15,
    xref="paper", yref="paper",
    text="Source: HDB Rental Data",
    showarrow=False,
    font=dict(size=10, color="gray"),
    xanchor="right"
)

fig.show()

# Save as high-quality PNG
fig.write_image(PLOTS_DIR / "year_over_year_rent_trends.png", scale=3)
print("✅ Year-over-year chart downloaded: year_over_year_rent_trends.png")

# Additional: Create a table summary for reference
print("\n📊 Summary of Year-over-Year Rent Trends:")
summary_table = yearly_complete.pivot_table(
    index='flat_type', 
    columns='year', 
    values='avg_rent', 
    aggfunc='mean'
).round(0)

print(summary_table)


In [ ]:
# Step 10: Downloading Charts

# Download all plots as HTML files
fig1.write_html("town_top_rental_volume.html")
fig2.write_html("town_rent_vs_volume.html")
fig3.write_html("town_flat_type_heatmap.html")
fig4.write_html("town_street_drilldown.html")
fig5.write_html("town_street_treemap.html")
fig6.write_html("town_rent_distribution.html")
fig7.write_html("town_analysis_dashboard.html")

print("All town/street charts have been saved as HTML files! You can open them in a browser and screenshot for PowerPoint.")


In [ ]:
# Step 11: Advanced EDA

print("Starting Advanced EDA...")

# 1. TIME SERIES DECOMPOSITION - Analyze seasonal patterns and trends in rental prices
print("1. Performing time series decomposition...")
from statsmodels.tsa.seasonal import seasonal_decompose

# Monthly average rent trend
monthly_avg = df_clean.set_index('rent_approval_date')['monthly_rent'].resample('M').mean()

# Decompose trend, seasonality, and residuals to understand patterns
decomposition = seasonal_decompose(monthly_avg.dropna(), model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(15, 12))
decomposition.observed.plot(ax=axes[0], title='Observed - Actual Rental Prices')
decomposition.trend.plot(ax=axes[1], title='Trend - Long-term Direction')
decomposition.seasonal.plot(ax=axes[2], title='Seasonal - Repeating Patterns')
decomposition.resid.plot(ax=axes[3], title='Residual - Unexplained Variation')
plt.tight_layout()
plt.show()

# Save decomposition plot
fig.savefig("time_series_decomposition.png", dpi=300, bbox_inches='tight')
print("✓ Time series decomposition completed")

# 2. MARKET SEGMENTATION & CLUSTERING - Group towns based on rental characteristics
print("2. Performing market segmentation clustering...")
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Create town-level features for clustering
town_features = df_clean.groupby('town').agg({
    'monthly_rent': ['mean', 'std', 'count'],
    'flat_type': lambda x: x.value_counts().index[0]  # most common flat type
}).round(2)
town_features.columns = ['avg_rent', 'rent_std', 'rental_count', 'dominant_flat_type']

# Encode categorical and scale numerical features for clustering
town_features_encoded = pd.get_dummies(town_features.reset_index(), columns=['dominant_flat_type'])
scaler = StandardScaler()
town_scaled = scaler.fit_transform(town_features_encoded.select_dtypes(include=[np.number]))

# K-means clustering to identify town segments
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
town_features['cluster'] = kmeans.fit_predict(town_scaled)

# Visualize town clusters
fig = px.scatter(town_features.reset_index(), x='avg_rent', y='rental_count', 
                 color='cluster', size='rent_std', hover_name='town',
                 title='Town Clusters Based on Rental Characteristics<br>Color=Cluster, Size=Price Variability')
fig.show()
fig.write_html("town_cluster_analysis.html")
print("✓ Market segmentation clustering completed")

# 3. MARKET CONCENTRATION ANALYSIS - Measure competition and market structure
print("3. Analyzing market concentration...")

def calculate_hhi(series):
    """Calculate Herfindahl-Hirschman Index for market concentration"""
    market_shares = series.value_counts(normalize=True) * 100
    return (market_shares ** 2).sum()

town_concentration = df_clean.groupby('town').agg({
    'monthly_rent': ['mean', 'count'],
    'flat_type': calculate_hhi,  # HHI for flat type diversity
    'street_name': calculate_hhi  # HHI for street concentration
})
town_concentration.columns = ['avg_rent', 'rental_count', 'flat_type_hhi', 'street_hhi']

# Visualize market concentration vs rent prices
fig = px.scatter(town_concentration.reset_index(), 
                 x='rental_count', y='avg_rent',
                 size='flat_type_hhi', color='street_hhi',
                 hover_name='town', 
                 title='Market Concentration vs Rent Prices<br>Size=Flat Type Diversity, Color=Street Concentration')
fig.show()
fig.write_html("market_concentration_analysis.html")
print("✓ Market concentration analysis completed")

# 4. ANOMALY DETECTION - Identify unusual rental prices (FIXED VERSION)
print("4. Detecting rental price anomalies...")
from sklearn.ensemble import IsolationForest

# Prepare features for anomaly detection - FIXED to avoid feature names warning
features_for_anomaly = df_clean[['monthly_rent']].copy()
features_for_anomaly['town_encoded'] = pd.factorize(df_clean['town'])[0]
features_for_anomaly['flat_type_encoded'] = pd.factorize(df_clean['flat_type'])[0]

# Convert to numpy array to avoid feature names warning
X_anomaly = features_for_anomaly.values

iso_forest = IsolationForest(contamination=0.05, random_state=42)
df_clean['is_anomaly'] = iso_forest.fit_predict(X_anomaly)

# Convert -1 (anomaly) to 1 and 1 (normal) to 0 for better visualization
df_clean['is_anomaly'] = df_clean['is_anomaly'].map({1: 0, -1: 1})

# Visualize detected anomalies
fig = px.scatter(df_clean, x='rent_approval_date', y='monthly_rent',
                 color='is_anomaly', hover_data=['town', 'flat_type'],
                 title='Anomaly Detection in Rental Prices<br>Red points indicate potential outliers',
                 color_continuous_scale=['blue', 'red'])
fig.show()
fig.write_html("anomaly_detection.html")
print("✓ Anomaly detection completed")

# 5. TEMPORAL PATTERN SIMILARITY - Compare rent patterns across towns
print("5. Analyzing temporal pattern similarity...")

# Create similarity matrix based on rental patterns (FIXED version)
# Convert Period index to string to avoid JSON serialization error
town_rent_matrix = pd.pivot_table(df_clean, 
                                  values='monthly_rent', 
                                  index='town', 
                                  columns=df_clean['rent_approval_date'].dt.to_period('M'),
                                  aggfunc='mean')

# Convert column names to strings to fix the Period serialization error
town_rent_matrix.columns = town_rent_matrix.columns.astype(str)

# Calculate correlation matrix between towns
town_correlation = town_rent_matrix.T.corr().fillna(0)  # Transpose to correlate towns

# Visualize town similarity heatmap
fig = px.imshow(town_correlation, 
                title='Temporal Rent Pattern Similarity Between Towns<br>Darker colors = More similar patterns',
                color_continuous_scale='RdBu_r',
                aspect='auto')
fig.update_layout(height=600)
fig.show()
fig.write_html("temporal_pattern_similarity.html")
print("✓ Temporal pattern similarity analysis completed")

# 6. RENT PREMIUM ANALYSIS - Calculate premium/discount vs town average
print("6. Analyzing rent premiums and discounts...")

# Calculate premium/discount for each rental vs town average
town_avg_rent = df_clean.groupby('town')['monthly_rent'].transform('mean')
df_clean['rent_premium'] = (df_clean['monthly_rent'] - town_avg_rent) / town_avg_rent * 100

# Analyze premium distribution by flat type
fig = px.box(df_clean, x='flat_type', y='rent_premium',
             title='Rent Premium/Discount Distribution by Flat Type<br>Positive values = Premium, Negative = Discount')
fig.show()
fig.write_html("rent_premium_analysis.html")
print("✓ Rent premium analysis completed")

# 7. RENT TREND FORECASTING - Predict future rent trends
print("7. Forecasting rent trends...")
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Simple linear trend forecasting for each town
forecast_results = []

for town in df_clean['town'].unique()[:10]:  # Top 10 towns by data availability
    town_data = df_clean[df_clean['town'] == town]
    monthly_trend = town_data.groupby(town_data['rent_approval_date'].dt.to_period('M'))['monthly_rent'].mean()
    
    if len(monthly_trend) > 6:  # Only towns with sufficient history
        X = np.arange(len(monthly_trend)).reshape(-1, 1)
        y = monthly_trend.values
        
        model = LinearRegression()
        model.fit(X, y)
        trend_slope = model.coef_[0]  # Monthly rent change
        
        forecast_results.append({
            'town': town,
            'trend_slope': trend_slope,
            'r_squared': r2_score(y, model.predict(X)),
            'current_rent': y[-1]
        })

forecast_df = pd.DataFrame(forecast_results)
fig = px.bar(forecast_df.nlargest(10, 'trend_slope'), 
             x='town', y='trend_slope',
             title='Top 10 Towns with Highest Rent Growth Trends<br>Positive slope = Increasing rents')
fig.show()
fig.write_html("rent_trend_forecasting.html")
print("✓ Rent trend forecasting completed")

# 8. PRICE SEGMENTATION ANALYSIS - Categorize rentals into market segments
print("8. Performing price segmentation analysis...")

# Analyze different price segments (Budget, Mid-Range, Premium, Luxury)
df_clean['rent_quantile'] = pd.qcut(df_clean['monthly_rent'], q=4, labels=['Budget', 'Mid-Range', 'Premium', 'Luxury'])

# Visualize market segments by town using sunburst chart
fig = px.sunburst(df_clean, path=['rent_quantile', 'town'],
                  values='monthly_rent', color='monthly_rent',
                  title='Rental Market Segments: Budget → Mid-Range → Premium → Luxury<br>Color intensity = Average rent')
fig.show()
fig.write_html("price_segmentation_analysis.html")
print("✓ Price segmentation analysis completed")

# 9. SEASONAL PATTERN ANALYSIS - Identify monthly rental patterns
print("9. Analyzing seasonal patterns...")

# Analyze rental patterns across different time periods
df_clean['month'] = df_clean['rent_approval_date'].dt.month
df_clean['quarter'] = df_clean['rent_approval_date'].dt.quarter

# Seasonal patterns by town and flat type
seasonal_patterns = df_clean.groupby(['town', 'flat_type', 'month']).agg({
    'monthly_rent': 'mean',
    'block': 'count'
}).reset_index()

fig = px.line(seasonal_patterns, x='month', y='monthly_rent',
              color='town', facet_col='flat_type',
              title='Seasonal Rent Patterns by Town and Flat Type<br>Lines show monthly rent fluctuations')
fig.update_layout(height=600)
fig.show()
fig.write_html("seasonal_patterns_analysis.html")
print("✓ Seasonal pattern analysis completed")


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# Create seasonal pattern analysis for each flat type
print("Analyzing seasonal patterns by flat type...")

# Extract month and year from rent_approval_date
df_clean['month'] = df_clean['rent_approval_date'].dt.month
df_clean['year'] = df_clean['rent_approval_date'].dt.year
df_clean['month_name'] = df_clean['rent_approval_date'].dt.month_name()

# Calculate monthly average rent for each flat type
seasonal_data = df_clean.groupby(['flat_type', 'year', 'month', 'month_name']).agg({
    'monthly_rent': ['mean', 'count', 'std']
}).round(2).reset_index()

# Flatten column names
seasonal_data.columns = ['flat_type', 'year', 'month', 'month_name', 'avg_rent', 'transaction_count', 'rent_std']

# Remove months with insufficient data (less than 10 transactions)
seasonal_data = seasonal_data[seasonal_data['transaction_count'] >= 10]

# Create month order for proper sorting
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']
seasonal_data['month_name'] = pd.Categorical(seasonal_data['month_name'], categories=month_order, ordered=True)

# Color palette for flat types
color_palette = {
    '1-ROOM': '#E74C3C',      # Red
    '2-ROOM': '#3498DB',      # Blue
    '3-ROOM': '#2ECC71',      # Green
    '4-ROOM': '#F39C12',      # Orange
    '5-ROOM': '#9B59B6',      # Purple
    'EXECUTIVE': '#1ABC9C'    # Teal
}

# Chart 1: Seasonal Patterns Over Years (Line Chart)
print("Creating seasonal patterns line chart...")

fig_lines = go.Figure()

for flat_type in sorted(seasonal_data['flat_type'].unique()):
    flat_data = seasonal_data[seasonal_data['flat_type'] == flat_type].sort_values(['year', 'month'])
    
    # Create a combined year-month label for x-axis
    flat_data['period'] = flat_data['year'].astype(str) + '-' + flat_data['month'].astype(str).str.zfill(2)
    
    fig_lines.add_trace(go.Scatter(
        x=flat_data['period'],
        y=flat_data['avg_rent'],
        mode='lines+markers',
        name=flat_type,
        line=dict(width=3, color=color_palette.get(flat_type, '#000000')),
        marker=dict(size=6),
        hovertemplate=(
            f"<b>{flat_type}</b><br>"
            "Period: %{x}<br>"
            "Avg Rent: $%{y:,.0f}<br>"
            "Transactions: %{customdata}<extra></extra>"
        ),
        customdata=flat_data['transaction_count']
    ))

fig_lines.update_layout(
    title=dict(
        text="<b>Seasonal Rental Patterns by Flat Type Over Time</b>",
        x=0.5,
        font=dict(size=20, color='#2C3E50')
    ),
    xaxis=dict(title="Time Period (Year-Month)", tickangle=45),
    yaxis=dict(title="Average Monthly Rent (S$)", tickformat="$,.0f"),
    height=700,
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    hovermode='closest'
)

fig_lines.show()

# Chart 2: Monthly Average Patterns (Box Plot Style)
print("Creating monthly average patterns chart...")

# Calculate overall monthly averages for each flat type
monthly_avg = seasonal_data.groupby(['flat_type', 'month', 'month_name']).agg({
    'avg_rent': 'mean',
    'transaction_count': 'sum'
}).reset_index()

fig_monthly = px.line(
    monthly_avg, 
    x='month_name', 
    y='avg_rent', 
    color='flat_type',
    color_discrete_map=color_palette,
    title="<b>Average Monthly Rental Patterns by Flat Type</b>",
    labels={'avg_rent': 'Average Monthly Rent (S$)', 'month_name': 'Month'},
    hover_data=['transaction_count']
)

fig_monthly.update_traces(
    line=dict(width=4),
    marker=dict(size=8)
)

fig_monthly.update_layout(
    height=600,
    xaxis=dict(categoryorder='array', categoryarray=month_order),
    yaxis=dict(tickformat="$,.0f"),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig_monthly.show()

# Chart 3: Seasonal Heatmap by Flat Type
print("Creating seasonal heatmap...")

# Prepare data for heatmap
heatmap_data = monthly_avg.pivot(index='flat_type', columns='month_name', values='avg_rent')
heatmap_data = heatmap_data[month_order]  # Ensure proper month order

fig_heatmap = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='Viridis',
    hoverongaps=False,
    hovertemplate=(
        "Flat Type: <b>%{y}</b><br>"
        "Month: <b>%{x}</b><br>"
        "Avg Rent: <b>$%{z:,.0f}</b><extra></extra>"
    )
))

fig_heatmap.update_layout(
    title=dict(
        text="<b>Seasonal Rent Patterns Heatmap</b><br><sub>Darker colors indicate higher rents</sub>",
        x=0.5,
        font=dict(size=18)
    ),
    height=500,
    xaxis=dict(title="Month"),
    yaxis=dict(title="Flat Type")
)

fig_heatmap.show()

# Chart 4: Seasonal Amplitude Analysis (Monthly Variations)
print("Creating seasonal amplitude analysis...")

# Calculate seasonal amplitude (max-min difference) for each flat type
seasonal_amplitude = monthly_avg.groupby('flat_type').agg({
    'avg_rent': ['min', 'max', 'mean']
}).round(2)
seasonal_amplitude.columns = ['min_rent', 'max_rent', 'mean_rent']
seasonal_amplitude['amplitude'] = seasonal_amplitude['max_rent'] - seasonal_amplitude['min_rent']
seasonal_amplitude['variation_pct'] = (seasonal_amplitude['amplitude'] / seasonal_amplitude['mean_rent'] * 100).round(1)

fig_amplitude = go.Figure()

for flat_type in seasonal_amplitude.index:
    data = seasonal_amplitude.loc[flat_type]
    
    fig_amplitude.add_trace(go.Bar(
        x=[flat_type],
        y=[data['amplitude']],
        name=flat_type,
        marker_color=color_palette.get(flat_type, '#000000'),
        hovertemplate=(
            f"<b>{flat_type}</b><br>"
            f"Seasonal Swing: ${data['amplitude']:,.0f}<br>"
            f"Variation: {data['variation_pct']}%<br>"
            f"Min: ${data['min_rent']:,.0f} | Max: ${data['max_rent']:,.0f}<extra></extra>"
        )
    ))

fig_amplitude.update_layout(
    title=dict(
        text="<b>Seasonal Rent Amplitude by Flat Type</b><br><sub>How much rents vary throughout the year</sub>",
        x=0.5,
        font=dict(size=18)
    ),
    xaxis=dict(title="Flat Type"),
    yaxis=dict(title="Seasonal Swing (S$)"),
    height=500,
    showlegend=False
)

fig_amplitude.show()

# Chart 5: Best/Worst Months for Each Flat Type
print("Creating best/worst months analysis...")

# Find best and worst months for each flat type
best_worst_months = []

for flat_type in monthly_avg['flat_type'].unique():
    flat_type_data = monthly_avg[monthly_avg['flat_type'] == flat_type]
    
    best_month = flat_type_data.loc[flat_type_data['avg_rent'].idxmax()]
    worst_month = flat_type_data.loc[flat_type_data['avg_rent'].idxmin()]
    
    best_worst_months.extend([
        {'flat_type': flat_type, 'month': best_month['month_name'], 'rent': best_month['avg_rent'], 'type': 'Best Month'},
        {'flat_type': flat_type, 'month': worst_month['month_name'], 'rent': worst_month['avg_rent'], 'type': 'Worst Month'}
    ])

best_worst_df = pd.DataFrame(best_worst_months)

fig_best_worst = px.scatter(
    best_worst_df,
    x='flat_type',
    y='rent',
    color='type',
    symbol='month',
    size_max=15,
    title="<b>Best and Worst Rental Months by Flat Type</b>",
    labels={'rent': 'Monthly Rent (S$)', 'flat_type': 'Flat Type'},
    hover_data=['month']
)

fig_best_worst.update_layout(
    height=600,
    yaxis=dict(tickformat="$,.0f"),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig_best_worst.show()

# Save all charts
fig_lines.write_image(PLOTS_DIR / "seasonal_patterns_timeline.png", scale=2)
fig_monthly.write_image(PLOTS_DIR / "monthly_average_patterns.png", scale=2)
fig_heatmap.write_image(PLOTS_DIR / "seasonal_heatmap.png", scale=2)
fig_amplitude.write_image(PLOTS_DIR / "seasonal_amplitude.png", scale=2)
fig_best_worst.write_image(PLOTS_DIR / "best_worst_months.png", scale=2)

print("\n✅ All seasonal pattern charts created successfully!")
print("📊 Charts saved:")
print("   - seasonal_patterns_timeline.png")
print("   - monthly_average_patterns.png") 
print("   - seasonal_heatmap.png")
print("   - seasonal_amplitude.png")
print("   - best_worst_months.png")

# Print key insights
print("\n🔍 Key Seasonal Insights:")
for flat_type in monthly_avg['flat_type'].unique():
    flat_data = monthly_avg[monthly_avg['flat_type'] == flat_type]
    best_month = flat_data.loc[flat_data['avg_rent'].idxmax()]
    worst_month = flat_data.loc[flat_data['avg_rent'].idxmin()]
    
    print(f"   {flat_type}:")
    print(f"     📈 Best: {best_month['month_name']} (${best_month['avg_rent']:,.0f})")
    print(f"     📉 Worst: {worst_month['month_name']} (${worst_month['avg_rent']:,.0f})")
    print(f"     📊 Seasonal Swing: ${(best_month['avg_rent'] - worst_month['avg_rent']):,.0f}")
    print()


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Create room count mapping with Executive as 6 rooms
room_mapping = {
    '1-ROOM': 1,
    '2-ROOM': 2,
    '3-ROOM': 3, 
    '4-ROOM': 4,
    '5-ROOM': 5,
    'EXECUTIVE': 6
}

# Apply room count mapping and ensure numeric type
spatial_df['room_count'] = spatial_df['flat_type'].map(room_mapping)
spatial_df['room_count'] = pd.to_numeric(spatial_df['room_count'], errors='coerce')

# Remove any rows with missing room counts or rent
rent_vs_rooms_df = spatial_df.dropna(subset=['room_count', 'monthly_rent'])

# Calculate Pearson correlation
correlation, p_value = pearsonr(rent_vs_rooms_df['room_count'], rent_vs_rooms_df['monthly_rent'])
r_squared = correlation ** 2

print(f"Pearson R: {correlation:.3f}")
print(f"R²: {r_squared:.3f}")
print(f"P-value: {p_value:.3e}")

# Create the scatter plot
fig = go.Figure()

# Add scatter points with some jitter for better visualization
np.random.seed(42)  # For reproducible jitter
jitter = np.random.normal(0, 0.1, len(rent_vs_rooms_df))

# Ensure room_count is numeric for the addition
room_count_numeric = rent_vs_rooms_df['room_count'].astype(float)

fig.add_trace(go.Scatter(
    x=room_count_numeric + jitter,
    y=rent_vs_rooms_df['monthly_rent'],
    mode='markers',
    marker=dict(
        size=6,
        color=rent_vs_rooms_df['monthly_rent'],
        colorscale='Viridis',
        opacity=0.6,
        showscale=True,
        colorbar=dict(title="Monthly Rent (S$)")
    ),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Rooms: %{customdata[1]}<br>"
        "Rent: $%{y:,.0f}<br>"
        "Town: %{customdata[2]}<br>"
        "<extra></extra>"
    ),
    customdata=np.column_stack([
        rent_vs_rooms_df['flat_type'],
        rent_vs_rooms_df['room_count'].astype(int).astype(str),
        rent_vs_rooms_df['town']
    ]),
    name="Rental Transactions"
))

# Add trend line
z = np.polyfit(room_count_numeric, rent_vs_rooms_df['monthly_rent'], 1)
p = np.poly1d(z)

trend_x = np.linspace(room_count_numeric.min(), room_count_numeric.max(), 100)
trend_y = p(trend_x)

fig.add_trace(go.Scatter(
    x=trend_x,
    y=trend_y,
    mode='lines',
    line=dict(color='red', width=3, dash='dash'),
    name=f'Trend Line (R² = {r_squared:.3f})'
))

# Update layout to match reference style
fig.update_layout(
    title=dict(
        text=f"<b>{r_squared:.3f}</b><br>"
             f"<span style='font-size:16px; color:gray'>Pearson R² between rent and room count</span><br><br>"
             f"<b>MONTHLY RENT</b><br>"
             f"<span style='font-size:14px; color:gray'>Executive apartments counted as 6 rooms</span>",
        x=0.5,
        xanchor='center',
        yanchor='top',
        font=dict(size=18)
    ),
    xaxis=dict(
        title="<b>ROOM COUNT</b>",
        tickmode='array',
        tickvals=list(range(1, 7)),
        ticktext=['1', '2', '3', '4', '5', '6 (Exec)'],
        gridcolor='lightgray',
        gridwidth=1
    ),
    yaxis=dict(
        title="<b>MONTHLY RENT (S$)</b>",
        tickformat="$,.0f",
        gridcolor='lightgray', 
        gridwidth=1
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=700,
    margin=dict(l=80, r=80, t=180, b=80),
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

# Add some styling
fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)

# Add annotation about data coverage
total_transactions = len(rent_vs_rooms_df)
coverage_pct = (len(rent_vs_rooms_df) / len(spatial_df)) * 100

fig.add_annotation(
    x=0.98, y=0.02,
    xref="paper", yref="paper",
    text=f"Data coverage: {coverage_pct:.1f}% ({total_transactions:,} transactions)",
    showarrow=False,
    font=dict(size=10, color="gray"),
    xanchor="right",
    bgcolor="white",
    bordercolor="lightgray",
    borderwidth=1
)

fig.show()

# Save as high-quality PNG
fig.write_image(PLOTS_DIR / "rent_vs_room_count_correlation.png", scale=3)
print("✅ Rent vs Room Count chart downloaded: rent_vs_room_count_correlation.png")

# Additional analysis: Show average rent by room count
print("\n📊 Average Rent by Room Count:")
avg_rent_by_rooms = rent_vs_rooms_df.groupby('room_count').agg({
    'monthly_rent': ['mean', 'median', 'count'],
    'flat_type': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown'
}).round(2)

avg_rent_by_rooms.columns = ['avg_rent', 'median_rent', 'transaction_count', 'most_common_type']
print(avg_rent_by_rooms)

# Create a bar chart showing average rent by room count
fig_bar = px.bar(
    avg_rent_by_rooms.reset_index(),
    x='room_count',
    y='avg_rent',
    text='avg_rent',
    title="<b>Average Monthly Rent by Room Count</b>",
    labels={'room_count': 'Room Count', 'avg_rent': 'Average Monthly Rent (S$)'},
    hover_data=['transaction_count', 'most_common_type']
)

fig_bar.update_traces(
    texttemplate='$%{y:,.0f}',
    textposition='outside',
    marker_color='#3498DB'
)

fig_bar.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=list(range(1, 7)),
        ticktext=['1 Room', '2 Room', '3 Room', '4 Room', '5 Room', 'Executive (6)']
    ),
    yaxis=dict(tickformat="$,.0f"),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500
)

fig_bar.show()
fig_bar.write_image(PLOTS_DIR / "avg_rent_by_room_count.png", scale=2)
print("✅ Average rent by room count chart downloaded: avg_rent_by_room_count.png")

# Print interpretation of the correlation
print(f"\n🔍 Interpretation:")
print(f"R² = {r_squared:.3f} means that {r_squared*100:.1f}% of the variation in rental prices")
print(f"can be explained by the number of rooms.")
print(f"This suggests room count has a {'strong' if r_squared > 0.5 else 'moderate' if r_squared > 0.3 else 'weak'} relationship with rent.")


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Create pie chart for town distribution
print("Creating town distribution pie chart...")

# Get town counts and percentages
town_counts = spatial_df['town'].value_counts()
total_transactions = len(spatial_df)
town_percentages = (town_counts / total_transactions * 100).round(1)

# Get date range for subtitle
if 'rent_approval_date' in spatial_df.columns and not spatial_df['rent_approval_date'].isna().all():
    date_range = f"{spatial_df['rent_approval_date'].min().strftime('%b %Y')} - {spatial_df['rent_approval_date'].max().strftime('%b %Y')}"
else:
    date_range = "Full Dataset"

# Create color palette - using a larger palette for towns
colors = [
    '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', 
    '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E9', '#F8C471', '#82E0AA',
    '#F1948A', '#85C1E9', '#D7BDE2', '#F9E79F', '#A9DFBF', '#F5B7B1',
    '#AED6F1', '#D2B4DE', '#ABEBC6', '#FAD7A0', '#BFC9CA', '#F5CBA7'
]

# Create the pie chart
fig_town_pie = go.Figure(data=[go.Pie(
    labels=town_counts.index,
    values=town_counts.values,
    hole=0.5,  # Creates the donut chart effect
    textinfo='label+percent',
    textposition='outside',
    marker=dict(colors=colors[:len(town_counts)]),
    hovertemplate='<b>%{label}</b><br>%{percent} (%{value:,} transactions)<extra></extra>',
    sort=False  # Keep original order (largest to smallest)
)])

# Update layout to match the flat type style
fig_town_pie.update_layout(
    title={
        'text': f"<b>{total_transactions:,} TRANSACTIONS</b><br>"
                f"<span style='font-size:14px; color:gray'>{date_range}</span>",
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 20}
    },
    showlegend=False,
    annotations=[
        dict(
            text=f"<b>{total_transactions:,}</b><br>TRANSACTIONS",
            x=0.5, y=0.5,
            font_size=16,
            showarrow=False,
            font_color='black'
        )
    ],
    height=700,  # Slightly taller to accommodate more labels
    margin=dict(t=100, b=50, l=50, r=50)
)

# Customize the text font and appearance
fig_town_pie.update_traces(
    textfont=dict(size=11, color='black'),  # Slightly smaller font for more towns
    texttemplate='%{label}<br>%{percent}%'
)

fig_town_pie.show()

# Save as PNG
fig_town_pie.write_image(PLOTS_DIR / "town_distribution_pie_chart.png", scale=2)
print("✅ Town distribution pie chart downloaded: town_distribution_pie_chart.png")

# Additional: Create a bar chart for top towns (more readable for many categories)
print("\nCreating complementary bar chart for top towns...")

# Show top 15 towns for better readability
top_towns = town_counts.head(15)

fig_town_bar = px.bar(
    x=top_towns.index,
    y=top_towns.values,
    title=f"<b>Top 15 Towns by Rental Transactions</b><br><span style='font-size:14px; color:gray'>{date_range}</span>",
    labels={'x': 'Town', 'y': 'Number of Transactions'},
    text=top_towns.values,
    color=top_towns.values,
    color_continuous_scale='Viridis'
)

fig_town_bar.update_traces(
    texttemplate='%{text:,}',
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Transactions: %{y:,}<extra></extra>'
)

fig_town_bar.update_layout(
    xaxis_tickangle=-45,
    yaxis_title="Number of Transactions",
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=600,
    showlegend=False,
    coloraxis_showscale=False
)

fig_town_bar.show()
fig_town_bar.write_image(PLOTS_DIR / "top_towns_bar_chart.png", scale=2)
print("✅ Top towns bar chart downloaded: top_towns_bar_chart.png")

# Print summary statistics
print(f"\n📊 Town Distribution Summary:")
print(f"Total towns: {len(town_counts)}")
print(f"Top 5 towns account for {town_percentages.head(5).sum():.1f}% of all transactions")
print(f"\nTop 10 Towns:")
for i, (town, count) in enumerate(town_counts.head(10).items(), 1):
    print(f"{i:2d}. {town:<20} {count:>6,} transactions ({town_percentages[town]:.1f}%)")

# Create a table of town statistics
town_stats = spatial_df.groupby('town').agg({
    'monthly_rent': ['mean', 'median', 'count'],
    'flat_type': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown'
}).round(2)

town_stats.columns = ['avg_rent', 'median_rent', 'transaction_count', 'most_common_type']
town_stats = town_stats.sort_values('transaction_count', ascending=False)

print(f"\n📈 Town Market Statistics (Top 10):")
print(town_stats.head(10).to_string())

# Optional: Create a sunburst chart for town + flat type combination
print("\nCreating sunburst chart for town and flat type combinations...")

# Get top towns and their flat type distribution
top_towns_list = town_counts.head(12).index.tolist()  # Top 12 towns
town_flat_data = spatial_df[spatial_df['town'].isin(top_towns_list)]

fig_sunburst = px.sunburst(
    town_flat_data,
    path=['town', 'flat_type'],
    title="<b>Rental Distribution: Town → Flat Type</b><br><sub>Interactive hierarchy showing market composition</sub>",
    height=700
)

fig_sunburst.show()
fig_sunburst.write_image(PLOTS_DIR / "town_flat_type_sunburst.png", scale=2)
print("✅ Town-flat type sunburst chart downloaded: town_flat_type_sunburst.png")
